In [924]:
import pandas as pd
from pathlib import Path
import os
import ast

In [925]:
#root = "/Volumes/ArxivE/_Tobias/LSM980/Mitotic_Stopwatch" + "/"
root = "Y:/_Tobias/LSM980/Mitotic_Stopwatch" + "/"
outputFolder = root + "DataFrames" + "/"

In [926]:
df = pd.read_csv(root + '/DataFrames/MainDataFrame_Lineages_Augmin.csv')
df.columns

Index(['Unnamed: 0', 'Track_ID', 'Source_ID', 'Splitting_event', 'Lineage',
       'Track_Coordinate_X', 'Track_Coordinate_Y', 'Frame', 'EndPoint_ID',
       'Position', 'Dataset', 'Condition', 'Generation', 'Mother_ID',
       'Grandmother_ID', 'Sister_ID', 'Daughters_ID',
       'Number_of_daughters_in_mitosis',
       'Mother_Number_of_daughters_in_mitosis', 'Cell_Cycle_mins',
       'Cell_Cycle_hours', 'Seen_sister', 'Seen_granny'],
      dtype='object')

In [927]:
# V2 error parsing and conversion


def concatenate_error_analysis_files(root_folder = root):

    root_folder = Path(root_folder)
    
    # Recursively find all matching files
    csv_files = list(root_folder.rglob("*_ErrorAnalysis.csv"))
    
    if not csv_files:
        print("No matching files found.")
        return pd.DataFrame()
    
    # Read and concatenate
    df_list = []
    for file in csv_files:
        try:
            df = pd.read_csv(file)
            df_list.append(df)
        except Exception as e:
            print(f"Error reading {file}: {e}")
    
    if df_list:
        df_errors_V2 = pd.concat(df_list, ignore_index = True)
    else:
        df_errors_V2 = pd.DataFrame()
    
    return df_errors_V2

def add_error_type_flags(df_errors_V2):
    error_types = [
       # "normal",
       # "blank",
        "Laggings",
        "Misaligned",
        "Cytokinesis Failure",
        "Massive Missegregation",
        "DNA bridge",
        "Multipolar Division",
        "Other",
        "Mitotic Death",
        "Slippage",
        "MN"
    ]
    
    for error in error_types:
        df_errors_V2[error] = (
            (df_errors_V2["1st Error Type"] == error) |
            (df_errors_V2["2nd Error Type"] == error)
        )
    
    # Create "any_true" column
    df_errors_V2["any_true"] = df_errors_V2[error_types].any(axis = 1)
    
    return df_errors_V2

df_errors_V2 = concatenate_error_analysis_files()
df_errors_V2 = add_error_type_flags(df_errors_V2)

df_errors_V2["Unique_Source_ID"] = df_errors_V2.Dataset.astype(str) + df_errors_V2.Position.astype(str) + df_errors_V2.Source_ID.astype(str)
print(df_errors_V2.Unique_Source_ID.nunique())
print(df_errors_V2.Dataset.unique())
df_errors_V2.head(20)

3666
[20260306 20260205 20260126 20251215 20251204]


,Unnamed: 0,Track_ID,Source_ID,Splitting_event,Lineage,Track_Coordinate_X,Track_Coordinate_Y,Frame,EndPoint_ID,Position,...,Cytokinesis Failure,Massive Missegregation,DNA bridge,Multipolar Division,Other,Mitotic Death,Slippage,MN,any_true,Unique_Source_ID
0,2,1,6107,True,6107.000,654.628,850.188,171,[],12,...,False,False,False,False,False,False,False,False,False,20260306126107
1,0,0,6052,True,6052.000,792.873,398.439,204,[],12,...,False,False,False,False,False,False,False,False,True,20260306126052
2,6,2,6186,True,6186.000,822.152,1001.968,276,[],12,...,False,False,False,False,False,False,False,False,False,20260306126186
3,3,1,6118,True,6107.612,823.257,772.295,405,[],12,...,False,False,False,False,False,False,False,False,False,20260306126118
4,9,2,6242,True,6186.624,698.961,878.362,500,[],12,...,False,False,False,False,False,False,False,False,False,20260306126242
5,7,2,6198,True,6186.620,772.710,1057.763,503,[],12,...,False,False,False,False,False,False,False,False,False,20260306126198
6,5,1,6158,True,6107.616,629.355,959.569,531,[],12,...,False,False,False,False,False,False,False,False,False,20260306126158
7,1,0,6068,True,6052.607,327.176,101.785,749,[],12,...,False,False,False,False,False,False,False,False,False,20260306126068
8,8,2,6211,True,6186.6198.6211,762.352,928.495,759,[],12,...,False,False,False,False,False,False,False,False,False,20260306126211
9,4,1,6132,True,6107.6118.6132,818.285,647.999,764,[],12,...,False,False,False,False,False,False,False,False,False,20260306126132


In [928]:
# Parse mitotic error tables (after FIJI analysis)
# Note: these are the first generation datasets only (see the error V2 analysis parsing above, for the remaining data)

error_path_1 = root + "Errors_siHAU6.xlsx"
error_path_2 = root + "Errors_siLUC1.xlsx"

error_paths = [error_path_1, error_path_2]

def parse_error(error_list = error_paths):
    dfs = []
    for path in error_list:
        df = pd.read_excel(path)
        dfs.append(df)
    return pd.concat(dfs)

df_error = parse_error().replace({"T": True, "F": False, "R": False})

# Fix dtypes in evaluation columns 
# (Mitotic death exluded from this list!)

cols = [
    "Laggings", 
    "Massive Missegregation", 
    "DNA bridge", 
    "Misaligned", 
    "Multipolar Division", 
    'Cytokinesis Failure',
    'Slippage',
    'MN' 
       ]

# Work on a copy to avoid warnings
bool_df = df_error.loc[:, cols].copy()

# Convert and cast to nullable boolean
bool_df = bool_df.apply(lambda col: col.astype("boolean"))
bool_df.dtypes

C:\Users\tkletter\AppData\Local\Temp\ipykernel_22000\299789669.py:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_error = parse_error().replace({"T": True, "F": False, "R": False})


Laggings                  boolean
Massive Missegregation    boolean
DNA bridge                boolean
Misaligned                boolean
Multipolar Division       boolean
Cytokinesis Failure       boolean
Slippage                  boolean
MN                        boolean
dtype: object

In [929]:
# Convert values: keep empty as NaN, only "True"/True as True, "False"/False as False
# TODO try to get this to work with the original df 
def to_bool_or_nan(x):
    if x in [True, "True"]:
        return True
    elif x in [False, "False"]:
        return False
    else:
        return np.nan   # preserve empties

#bool_df = bool_df.map(to_bool_or_nan)
bool_df.loc[:, "any_true"] = bool_df.any(axis = 1, bool_only = True)

# New column: True if any True, NaN if all are NaN, otherwise False
df_error.loc[:, "any_true"] = bool_df.any(axis = 1)

df_error.to_csv(outputFolder + "Concat_anytrue.csv")

In [930]:
df_error.columns

Index(['Source_ID', 'Dataset', 'Position', 'Laggings',
       'Massive Missegregation', 'DNA bridge', 'Misaligned', 'MN',
       'Multipolar Division', 'Cytokinesis Failure', 'Slippage',
       'Mitotic Death', 'Comments', 'any_true'],
      dtype='object')

In [931]:
# cropping V2 errors before concatenating vertically

df_errors_V2 = df_errors_V2[['Source_ID', 'Dataset', 'Position', 'Laggings',
       'Massive Missegregation', 'DNA bridge', 'Misaligned', 'MN',
       'Multipolar Division', 'Cytokinesis Failure', 'Slippage',
       'Mitotic Death', 'any_true']]

In [932]:
df_error = pd.concat([df_error, df_errors_V2])

df_error.to_csv(outputFolder + "Concat_anytrue_V2.csv")

In [933]:
# parse Death dataframes after daughter analysis (FIJI)

def getdeaths(filepath, position, dataset):
    print(filepath)
    deaths = pd.read_csv(filepath, usecols = ["Death", "EndPoint_ID", "Track_ID", "Downstream_IDs"])
    deaths["Position"] = position
    deaths["Dataset"] = dataset
    
    return deaths

# Define metadata for each file
deaths_info = [
    {"date": "20250724", "position": 2, "target": "LUC1", "file": "20250724_IM_H2B-GFP_MitoStop_RNAi03_downstream_Filled.csv"},
    {"date": "20250724", "position": 3, "target": "LUC1", "file": "20250724_IM_H2B-GFP_MitoStop_RNAi03_downstream_Filled.csv"},
    {"date": "20250724", "position": 4, "target": "LUC1", "file": "20250724_IM_H2B-GFP_MitoStop_RNAi03_downstream_Filled.csv"},
    {"date": "20250724", "position": 5, "target": "LUC1", "file": "20250724_IM_H2B-GFP_MitoStop_RNAi03_downstream_Filled.csv"},
    {"date": "20250724", "position": 6, "target": "HAUS6", "file": "20250724_IM_H2B-GFP_MitoStop_RNAi03_downstream_Filled.csv"},
    {"date": "20250724", "position": 7, "target": "HAUS6", "file": "20250724_IM_H2B-GFP_MitoStop_RNAi03_downstream_Filled.csv"},
    {"date": "20250724", "position": 8, "target": "HAUS6", "file": "20250724_IM_H2B-GFP_MitoStop_RNAi03_downstream_Filled.csv"},
    {"date": "20250724", "position": 9, "target": "HAUS6", "file": "20250724_IM_H2B-GFP_MitoStop_RNAi03_downstream_Filled.csv"},
    
    {"date": "20250807", "position": 2, "target": "LUC1", "file": "20250807_IM_H2B-GFP_MitoStop_RNAi04_downstream_Filled.csv"},
    {"date": "20250807", "position": 3, "target": "LUC1", "file": "20250807_IM_H2B-GFP_MitoStop_RNAi04_downstream_Filled.csv"},
    {"date": "20250807", "position": 4, "target": "LUC1", "file": "20250807_IM_H2B-GFP_MitoStop_RNAi04_downstream_Filled.csv"},
    {"date": "20250807", "position": 5, "target": "HAUS6", "file": "20250807_IM_H2B-GFP_MitoStop_RNAi04_downstream_Filled.csv"},
    {"date": "20250807", "position": 6, "target": "HAUS6", "file": "20250807_IM_H2B-GFP_MitoStop_RNAi04_downstream_Filled.csv"},
    {"date": "20250807", "position": 7, "target": "HAUS6", "file": "20250807_IM_H2B-GFP_MitoStop_RNAi04_downstream_Filled.csv"},
    {"date": "20250807", "position": 8, "target": "HAUS6", "file": "20250807_IM_H2B-GFP_MitoStop_RNAi04_downstream_Filled.csv"},
    {"date": "20250807", "position": 9, "target": "HAUS6", "file": "20250807_IM_H2B-GFP_MitoStop_RNAi04_downstream_Filled.csv"},

    {"date": "20250814", "position": 2, "target": "LUC1", "file": "20250814_IM_H2B-GFP_MitoStop_RNAi05_downstream_Filled.csv"},
    {"date": "20250814", "position": 3, "target": "LUC1", "file": "20250814_IM_H2B-GFP_MitoStop_RNAi05_downstream_Filled.csv"},
    {"date": "20250814", "position": 4, "target": "LUC1", "file": "20250814_IM_H2B-GFP_MitoStop_RNAi05_downstream_Filled.csv"},
    {"date": "20250814", "position": 5, "target": "HAUS6", "file": "20250814_IM_H2B-GFP_MitoStop_RNAi05_downstream_Filled.csv"},
    {"date": "20250814", "position": 6, "target": "HAUS6", "file": "20250814_IM_H2B-GFP_MitoStop_RNAi05_downstream_Filled.csv"},
    {"date": "20250814", "position": 7, "target": "HAUS6", "file": "20250814_IM_H2B-GFP_MitoStop_RNAi05_downstream_Filled.csv"},
    {"date": "20250814", "position": 8, "target": "HAUS6", "file": "20250814_IM_H2B-GFP_MitoStop_RNAi05_downstream_Filled.csv"},
   # {"date": "20250814", "position": 9, "target": "HAUS6", "file": "20250807_IM_H2B-GFP_MitoStop_RNAi04_onsets.csv"},
]

deaths_info2 = [
    {"date": "20260126_USP28", "position": "01", "target": "siLUC1", "file": "20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_downstream_Filled.csv"},
    {"date": "20260126_USP28", "position": "02", "target": "siLUC1", "file": "20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_downstream_Filled.csv"},
    {"date": "20260126_USP28", "position": "03", "target": "siLUC1siHAUS6", "file": "20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_downstream_Filled.csv"},
    {"date": "20260126_USP28", "position": "04", "target": "siLUC1siHAUS6", "file": "20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_downstream_Filled.csv"},
    {"date": "20260126_USP28", "position": "05", "target": "siLUC1siHAUS6", "file": "20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_downstream_Filled.csv"},
    {"date": "20260126_USP28", "position": "06", "target": "siLUC1siHAUS6", "file": "20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_downstream_Filled.csv"},
    {"date": "20260126_USP28", "position": "07", "target": "siLUC1siHAUS6", "file": "20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_downstream_Filled.csv"},
    {"date": "20260126_USP28", "position": "08", "target": "siUSP28siHAUS6", "file": "20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_downstream_Filled.csv"},
    {"date": "20260126_USP28", "position": "09", "target": "siUSP28siHAUS6", "file": "20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_downstream_Filled.csv"},
    {"date": "20260126_USP28", "position": "10", "target": "siUSP28siHAUS6", "file": "20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_downstream_Filled.csv"},
    {"date": "20260126_USP28", "position": "11", "target": "siUSP28siHAUS6", "file": "20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_downstream_Filled.csv"},
    {"date": "20260126_USP28", "position": "12", "target": "siUSP28siHAUS6", "file": "20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_downstream_Filled.csv"},
    {"date": "20260126_USP28", "position": "13", "target": "siUSP28siHAUS6", "file": "20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_downstream_Filled.csv"},
    {"date": "20260126_USP28", "position": "14", "target": "siUSP28siHAUS6", "file": "20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_downstream_Filled.csv"},

    {"date": "20260205_USP28", "position": "01", "target": "siLUC", "file": "20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_downstream_Filled.csv"},
    {"date": "20260205_USP28", "position": "02", "target": "siLUC", "file": "20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_downstream_Filled.csv"},
    {"date": "20260205_USP28", "position": "03", "target": "siLUCsiHAUS6", "file": "20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_downstream_Filled.csv"},
    {"date": "20260205_USP28", "position": "04", "target": "siLUCsiHAUS6", "file": "20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_downstream_Filled.csv"},
    {"date": "20260205_USP28", "position": "05", "target": "siLUCsiHAUS6", "file": "20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_downstream_Filled.csv"},
    {"date": "20260205_USP28", "position": "06", "target": "siLUCsiHAUS6", "file": "20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_downstream_Filled.csv"},
    {"date": "20260205_USP28", "position": "07", "target": "siLUCsiHAUS6", "file": "20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_downstream_Filled.csv"},
    {"date": "20260205_USP28", "position": "08", "target": "siUSP28HAUS6", "file": "20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_downstream_Filled.csv"},
    {"date": "20260205_USP28", "position": "09", "target": "siUSP28HAUS6", "file": "20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_downstream_Filled.csv"},
    {"date": "20260205_USP28", "position": "10", "target": "siUSP28HAUS6", "file": "20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_downstream_Filled.csv"},
    {"date": "20260205_USP28", "position": "11", "target": "siUSP28HAUS6", "file": "20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_downstream_Filled.csv"},
    {"date": "20260205_USP28", "position": "12", "target": "siUSP28HAUS6", "file": "20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_downstream_Filled.csv"},
    {"date": "20260205_USP28", "position": "13", "target": "siUSP28HAUS6", "file": "20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_downstream_Filled.csv"},
    {"date": "20260205_USP28", "position": "14", "target": "siUSP28HAUS6", "file": "20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_downstream_Filled.csv"},

    {"date": "20260306_USP28", "position": "01", "target": "siLUC1", "file": "20260306_IM_H2B-GFP_MitoStop_RNAiUSP28_downstream_Filled.csv"},
    {"date": "20260306_USP28", "position": "02", "target": "siLUC1", "file": "20260306_IM_H2B-GFP_MitoStop_RNAiUSP28_downstream_Filled.csv"},
    {"date": "20260306_USP28", "position": "03", "target": "siLUC1", "file": "20260306_IM_H2B-GFP_MitoStop_RNAiUSP28_downstream_Filled.csv"},
    {"date": "20260306_USP28", "position": "04", "target": "siLUC1", "file": "20260306_IM_H2B-GFP_MitoStop_RNAiUSP28_downstream_Filled.csv"},
    {"date": "20260306_USP28", "position": "05", "target": "siLUC1siHAUS6", "file": "20260306_IM_H2B-GFP_MitoStop_RNAiUSP28_downstream_Filled.csv"},
    {"date": "20260306_USP28", "position": "06", "target": "siLUC1siHAUS6", "file": "20260306_IM_H2B-GFP_MitoStop_RNAiUSP28_downstream_Filled.csv"},
    {"date": "20260306_USP28", "position": "07", "target": "siLUC1siHAUS6", "file": "20260306_IM_H2B-GFP_MitoStop_RNAiUSP28_downstream_Filled.csv"},
    {"date": "20260306_USP28", "position": "08", "target": "siUSP28siHAUS6", "file": "20260306_IM_H2B-GFP_MitoStop_RNAiUSP28_downstream_Filled.csv"},
    {"date": "20260306_USP28", "position": "09", "target": "siUSP28siHAUS6", "file": "20260306_IM_H2B-GFP_MitoStop_RNAiUSP28_downstream_Filled.csv"},
    {"date": "20260306_USP28", "position": "10", "target": "siUSP28siHAUS6", "file": "20260306_IM_H2B-GFP_MitoStop_RNAiUSP28_downstream_Filled.csv"},
    {"date": "20260306_USP28", "position": "11", "target": "siUSP28siHAUS6", "file": "20260306_IM_H2B-GFP_MitoStop_RNAiUSP28_downstream_Filled.csv"},
   # {"date": "20260306_USP28", "position": "12", "target": "siUSP28siHAUS6", "file": "20260306_IM_H2B-GFP_MitoStop_RNAiUSP28_downstream_Filled.csv"}, # empty
]


# Generate and combine dataframes
df_deaths_list = []

for info in deaths_info:
    folder = f"{root}/{info['date']}/Position_{info['position']}_si{info['target']}/Concatenated"
    filepath = os.path.join(folder, info["file"])
    df_single = getdeaths(filepath, info["position"], info["date"])
    df_deaths_list.append(df_single)

for info in deaths_info2:
    base_folder = f"{root}/{info['date']}/Position_{info['position']}_{info['target']}"

    folder = None
    for f in os.listdir(base_folder):
        if f.lower() == "max":
            folder = os.path.join(base_folder, f)
            break

    filepath = os.path.join(folder, info["file"])
    df_single = getdeaths(filepath, info["position"], info["date"])
    
    df_deaths_list.append(df_single)

df_deaths_proto = pd.concat(df_deaths_list, ignore_index = True)
#df_deaths["Dataset"] = df_deaths.Dataset.astype(int)
#df_deaths["Dataset"] = df_deaths.Dataset.astype(int)
df_deaths_proto["EndPoint_ID"] = df_deaths_proto.EndPoint_ID.astype(str)
df_deaths_proto["Death"] = df_deaths_proto.Death.astype(str)

df_deaths_proto["Dataset"] = df_deaths_proto.Dataset.str[:8]

df_deaths_proto.to_csv(outputFolder + "Debug_Deaths_Proto.csv")

print(df_deaths_proto.shape)
df_deaths = df_deaths_proto[(df_deaths_proto["Death"] == "TRUE")]
print(df_deaths.shape)


df_leaves = df_deaths_proto[(df_deaths_proto["Death"] == "FALSE")]
print(df_leaves.shape)



df_deaths["Death"] = df_deaths["Death"].str.lower().map({"true": True, "false": False})
df_leaves["Leaves"] = df_leaves["Death"].str.lower().map({"true": False, "false": True})

df_deaths.to_csv(outputFolder + "Debug_Deaths.csv")
df_leaves.to_csv(outputFolder + "Debug_Leaves.csv")

df["EndPoint_ID"] = df["EndPoint_ID"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

# Collect all endpoint IDs from df (flatten the lists)
all_df_endpoints = set(e for lst in df["EndPoint_ID"] if isinstance(lst, list) for e in lst)

# Collect all death endpoint IDs
all_dead_endpoints = set(df_deaths.loc[df_deaths["Death"] == True, "EndPoint_ID"])

# Collect all Leaves endpoint IDs
all_leaves_endpoints = set(df_leaves.loc[df_leaves["Leaves"] == True, "EndPoint_ID"])

print("Number of endpoints in df:", len(all_df_endpoints))
print("Number of death endpoints:", len(all_dead_endpoints))
print("Number of death overlaps:", len(all_df_endpoints & all_dead_endpoints))
print("Number of leaves endpoints:", len(all_leaves_endpoints))
print("Number of leaves overlaps:", len(all_df_endpoints & all_leaves_endpoints))

Y:/_Tobias/LSM980/Mitotic_Stopwatch//20250724/Position_2_siLUC1/Concatenated\20250724_IM_H2B-GFP_MitoStop_RNAi03_downstream_Filled.csv
Y:/_Tobias/LSM980/Mitotic_Stopwatch//20250724/Position_3_siLUC1/Concatenated\20250724_IM_H2B-GFP_MitoStop_RNAi03_downstream_Filled.csv
Y:/_Tobias/LSM980/Mitotic_Stopwatch//20250724/Position_4_siLUC1/Concatenated\20250724_IM_H2B-GFP_MitoStop_RNAi03_downstream_Filled.csv
Y:/_Tobias/LSM980/Mitotic_Stopwatch//20250724/Position_5_siLUC1/Concatenated\20250724_IM_H2B-GFP_MitoStop_RNAi03_downstream_Filled.csv
Y:/_Tobias/LSM980/Mitotic_Stopwatch//20250724/Position_6_siHAUS6/Concatenated\20250724_IM_H2B-GFP_MitoStop_RNAi03_downstream_Filled.csv
Y:/_Tobias/LSM980/Mitotic_Stopwatch//20250724/Position_7_siHAUS6/Concatenated\20250724_IM_H2B-GFP_MitoStop_RNAi03_downstream_Filled.csv
Y:/_Tobias/LSM980/Mitotic_Stopwatch//20250724/Position_8_siHAUS6/Concatenated\20250724_IM_H2B-GFP_MitoStop_RNAi03_downstream_Filled.csv
Y:/_Tobias/LSM980/Mitotic_Stopwatch//20250724/Positi

C:\Users\tkletter\AppData\Local\Temp\ipykernel_22000\3336207224.py:130: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_deaths["Death"] = df_deaths["Death"].str.lower().map({"true": True, "false": False})
C:\Users\tkletter\AppData\Local\Temp\ipykernel_22000\3336207224.py:131: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_leaves["Leaves"] = df_leaves["Death"].str.lower().map({"true": False, "false": True})


In [934]:
# Find a way to only look at Generation == 1


#
#def count_dead_offspring(row, df_deaths):
#    # Filter deaths for same Track_ID & Dataset
#    deaths_subset = df_deaths[
#        (df_deaths["Track_ID"] == row["Track_ID"]) &
#        (df_deaths["Dataset"] == row["Dataset"]) &
#        (df_deaths["Death"])
#    ]
#    dead_endpoints = set(deaths_subset["EndPoint_ID"])
#    print(dead_endpoints)
#    
#    # Count how many of this row's endpoints are in the dead set
#    endpoint_list = row["EndPoint_ID"]
#    if isinstance(endpoint_list, list):
#        return sum(e in dead_endpoints for e in endpoint_list)
#    return 0

#def count_dead_offspring(row, df_deaths):
#
#    print("Row Track_ID:", row["Track_ID"])
#    print("Row Dataset:", row["Dataset"])
#
#    deaths_subset = df_deaths[
#        (df_deaths["Track_ID"] == row["Track_ID"]) &
#        (df_deaths["Dataset"] == row["Dataset"]) &
#        (df_deaths["Death"])
#    ]
#
#    print("Subset rows:", len(deaths_subset))
#
#    dead_endpoints = set(deaths_subset["EndPoint_ID"])
#    print("Dead endpoints:", dead_endpoints)
#
#    endpoint_list = row["EndPoint_ID"]
#    if isinstance(endpoint_list, list):
#        return sum(e in dead_endpoints for e in endpoint_list)
#
#    return 0
#df["N_dead_offspring"] = df.apply(count_dead_offspring, axis = 1, df_deaths = df_deaths)

print(df_deaths["Death"].dtype)
print(df_deaths["Death"].unique())
print(df["Track_ID"].dtype)
print(df_deaths["Track_ID"].dtype)

df["Dataset"] = df["Dataset"].astype(str)

print(df["Dataset"].dtype)
print(df_deaths["Dataset"].dtype)
print(df["Dataset"].unique())
print(df_deaths["Dataset"].unique())


dead_map = (
    df_deaths[df_deaths["Death"]]
    .groupby(["Track_ID", "Dataset"])["EndPoint_ID"]
    .apply(set)
)

leaves_map = (
    df_leaves[df_leaves["Leaves"]]
    .groupby(["Track_ID", "Dataset"])["EndPoint_ID"]
    .apply(set)
)

bool
[ True]
int64
int64
object
object
['20250724' '20250807' '20250814' '20251204' '20251215' '20260126'
 '20260205' '20260306']
['20260126' '20260205' '20260306']


In [935]:
def count_dead_offspring(row):
    dead_endpoints = dead_map.get((row["Track_ID"], row["Dataset"]), set())
    endpoints = row["EndPoint_ID"]

    if isinstance(endpoints, list):
        return sum(e in dead_endpoints for e in endpoints)
    return 0

df["N_dead_offspring"] = df.apply(count_dead_offspring, axis=1)

In [936]:
def count_leaving_offspring(row):
    leaves_endpoints = leaves_map.get((row["Track_ID"], row["Dataset"]), set())
    endpoints = row["EndPoint_ID"]

    if isinstance(endpoints, list):
        return sum(e in leaves_endpoints for e in endpoints)
    return 0

df["N_leaving_offspring"] = df.apply(count_leaving_offspring, axis=1)

In [937]:
# Find a way to only look at Generation == 1

#def count_leaving_offspring(row, df_leaves):
#    # Filter leaves for same Track_ID & Dataset
#    leaves_subset = df_leaves[
#        (df_leaves["Track_ID"] == row["Track_ID"]) &
#        (df_leaves["Dataset"] == row["Dataset"]) &
#        (df_leaves["Death"])
#    ]
#    leaves_endpoints = set(leaves_subset["EndPoint_ID"])
#
#    # Count how many of this row's endpoints are in the leaves set
#    endpoint_list2 = row["EndPoint_ID"]
#    if isinstance(endpoint_list2, list):
#        return sum(e in leaves_endpoints for e in endpoint_list2)
#    return 0
#
#df["N_leaving_offspring"] = df.apply(count_leaving_offspring, axis = 1, df_leaves = df_leaves)

In [938]:
df_error["Unique_Source_ID"] = df_error.Dataset.astype(str) + df_error.Position.astype(str) + df_error.Source_ID.astype(str)
# Final (outer) merge with error dataframe

# df_outer = df.merge(df_error, how = "outer")
#df_outer.to_csv(outputFolder + "MainDataFrame_MitoticStopwatch_Augmin_before_merge.csv")

# Final (inner) merge with error dataframe to only include evaluated divisions
df_error["Dataset"] = df_error["Dataset"].astype(str)
df = df.merge(df_error, how = "inner")

df.to_csv(outputFolder + "Debug_before_merge_with_Onsets.csv")

In [939]:
# TODO make this as elegant as the errors V2 parsing

def getonsets(filepath, position, dataset):
    onsets = pd.read_csv(filepath)
    onsets["Source_ID"] = onsets.Label.str.split(":").str.get(1).astype(int)
    onsets["Position"] = position
    onsets["Dataset"] = dataset
    onsets.rename(columns = {"Slice": "Frame_onset"}, inplace = True)
    onsets["Frame_onset"] = onsets.Frame_onset - 1 # this corrects for the +1 frame set by FIJI
    return onsets

# Define metadata for each file
onsets_info = [
    {"date": "20250724", "position": 2, "target": "LUC1", "file": "20250724_IM_H2B-GFP_MitoStop_RNAi03_onsets.csv"},
    {"date": "20250724", "position": 3, "target": "LUC1", "file": "20250724_IM_H2B-GFP_MitoStop_RNAi03_onsets.csv"},
    {"date": "20250724", "position": 4, "target": "LUC1", "file": "20250724_IM_H2B-GFP_MitoStop_RNAi03_onsets.csv"},
    {"date": "20250724", "position": 5, "target": "LUC1", "file": "20250724_IM_H2B-GFP_MitoStop_RNAi03_onsets.csv"},
    {"date": "20250724", "position": 6, "target": "HAUS6", "file": "20250724_IM_H2B-GFP_MitoStop_RNAi03_onsets.csv"},
    {"date": "20250724", "position": 7, "target": "HAUS6", "file": "20250724_IM_H2B-GFP_MitoStop_RNAi03_onsets.csv"},
    {"date": "20250724", "position": 8, "target": "HAUS6", "file": "20250724_IM_H2B-GFP_MitoStop_RNAi03_onsets.csv"},
    {"date": "20250724", "position": 9, "target": "HAUS6", "file": "20250724_IM_H2B-GFP_MitoStop_RNAi03_onsets.csv"},
    
    {"date": "20250807", "position": 2, "target": "LUC1", "file": "20250807_IM_H2B-GFP_MitoStop_RNAi04_onsets.csv"},
    {"date": "20250807", "position": 3, "target": "LUC1", "file": "20250807_IM_H2B-GFP_MitoStop_RNAi04_onsets.csv"},
    {"date": "20250807", "position": 4, "target": "LUC1", "file": "20250807_IM_H2B-GFP_MitoStop_RNAi04_onsets.csv"},
    {"date": "20250807", "position": 5, "target": "HAUS6", "file": "20250807_IM_H2B-GFP_MitoStop_RNAi04_onsets.csv"},
    {"date": "20250807", "position": 6, "target": "HAUS6", "file": "20250807_IM_H2B-GFP_MitoStop_RNAi04_onsets.csv"},
    {"date": "20250807", "position": 7, "target": "HAUS6", "file": "20250807_IM_H2B-GFP_MitoStop_RNAi04_onsets.csv"},
    {"date": "20250807", "position": 8, "target": "HAUS6", "file": "20250807_IM_H2B-GFP_MitoStop_RNAi04_onsets.csv"},
    {"date": "20250807", "position": 9, "target": "HAUS6", "file": "20250807_IM_H2B-GFP_MitoStop_RNAi04_onsets.csv"},

    {"date": "20250814", "position": 2, "target": "LUC1", "file": "20250814_IM_H2B-GFP_MitoStop_RNAi05_onsets.csv"},
    {"date": "20250814", "position": 3, "target": "LUC1", "file": "20250814_IM_H2B-GFP_MitoStop_RNAi05_onsets.csv"},
    {"date": "20250814", "position": 4, "target": "LUC1", "file": "20250814_IM_H2B-GFP_MitoStop_RNAi05_onsets.csv"},
    {"date": "20250814", "position": 5, "target": "HAUS6", "file": "20250814_IM_H2B-GFP_MitoStop_RNAi05_onsets.csv"},
    {"date": "20250814", "position": 6, "target": "HAUS6", "file": "20250814_IM_H2B-GFP_MitoStop_RNAi05_onsets.csv"},
    {"date": "20250814", "position": 7, "target": "HAUS6", "file": "20250814_IM_H2B-GFP_MitoStop_RNAi05_onsets.csv"},
    {"date": "20250814", "position": 8, "target": "HAUS6", "file": "20250814_IM_H2B-GFP_MitoStop_RNAi05_onsets.csv"},
   ### {"date": "20250814", "position": 9, "target": "HAUS6", "file": "20250807_IM_H2B-GFP_MitoStop_RNAi04_onsets.csv"},
]

#### Second generation data sets (HAUS6 only) and USP28 double depletion
# Define metadata for each file
onsets_info2 = [
    {"date": "20251204", "position": "01", "target": "siLUC1", "file": "20251204_IM_H2B-GFP_MitoStop_RNAi201_onsets.csv"},
    {"date": "20251204", "position": "02", "target": "siLUC1", "file": "20251204_IM_H2B-GFP_MitoStop_RNAi201_onsets.csv"},
    {"date": "20251204", "position": "03", "target": "siLUC1", "file": "20251204_IM_H2B-GFP_MitoStop_RNAi201_onsets.csv"},
    {"date": "20251204", "position": "04", "target": "siHAUS6", "file": "20251204_IM_H2B-GFP_MitoStop_RNAi201_onsets.csv"},
    {"date": "20251204", "position": "05", "target": "siHAUS6", "file": "20251204_IM_H2B-GFP_MitoStop_RNAi201_onsets.csv"},
    {"date": "20251204", "position": "06", "target": "siHAUS6", "file": "20251204_IM_H2B-GFP_MitoStop_RNAi201_onsets.csv"},
    {"date": "20251204", "position": "07", "target": "siHAUS6", "file": "20251204_IM_H2B-GFP_MitoStop_RNAi201_onsets.csv"},
    {"date": "20251204", "position": "08", "target": "siHAUS6", "file": "20251204_IM_H2B-GFP_MitoStop_RNAi201_onsets.csv"},
    {"date": "20251204", "position": "09", "target": "siHAUS6", "file": "20251204_IM_H2B-GFP_MitoStop_RNAi201_onsets.csv"},
    {"date": "20251204", "position": "10", "target": "siHAUS6", "file": "20251204_IM_H2B-GFP_MitoStop_RNAi201_onsets.csv"},
    {"date": "20251204", "position": "11", "target": "siHAUS6", "file": "20251204_IM_H2B-GFP_MitoStop_RNAi201_onsets.csv"},
    
   {"date": "20251215", "position": "01", "target": "Mock", "file": "20251215_IM_H2B-GFP_MitoStop_RNAi202_onsets.csv"},
    {"date": "20251215", "position": "02", "target": "Mock", "file": "20251215_IM_H2B-GFP_MitoStop_RNAi202_onsets.csv"},
    {"date": "20251215", "position": "03", "target": "siLUC1", "file": "20251215_IM_H2B-GFP_MitoStop_RNAi202_onsets.csv"},
    {"date": "20251215", "position": "04", "target": "siLUC1", "file": "20251215_IM_H2B-GFP_MitoStop_RNAi202_onsets.csv"},
    {"date": "20251215", "position": "05", "target": "siLUC1", "file": "20251215_IM_H2B-GFP_MitoStop_RNAi202_onsets.csv"},
    {"date": "20251215", "position": "06", "target": "siLUC1", "file": "20251215_IM_H2B-GFP_MitoStop_RNAi202_onsets.csv"},
    {"date": "20251215", "position": "07", "target": "siHAUS6", "file": "20251215_IM_H2B-GFP_MitoStop_RNAi202_onsets.csv"},
    {"date": "20251215", "position": "08", "target": "siHAUS6", "file": "20251215_IM_H2B-GFP_MitoStop_RNAi202_onsets.csv"},
    {"date": "20251215", "position": "09", "target": "siHAUS6", "file": "20251215_IM_H2B-GFP_MitoStop_RNAi202_onsets.csv"},
    {"date": "20251215", "position": "10", "target": "siHAUS6", "file": "20251215_IM_H2B-GFP_MitoStop_RNAi202_onsets.csv"},
    {"date": "20251215", "position": "11", "target": "siHAUS6", "file": "20251215_IM_H2B-GFP_MitoStop_RNAi202_onsets.csv"},
    {"date": "20251215", "position": "12", "target": "siHAUS6", "file": "20251215_IM_H2B-GFP_MitoStop_RNAi202_onsets.csv"},
    {"date": "20251215", "position": "13", "target": "siHAUS6", "file": "20251215_IM_H2B-GFP_MitoStop_RNAi202_onsets.csv"},
    {"date": "20251215", "position": "14", "target": "siHAUS6", "file": "20251215_IM_H2B-GFP_MitoStop_RNAi202_onsets.csv"},
    
    {"date": "20260126_USP28", "position": "01", "target": "siLUC1", "file": "20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_onsets.csv"},
    {"date": "20260126_USP28", "position": "02", "target": "siLUC1", "file": "20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_onsets.csv"},
    {"date": "20260126_USP28", "position": "03", "target": "siLUC1siHAUS6", "file": "20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_onsets.csv"},
    {"date": "20260126_USP28", "position": "04", "target": "siLUC1siHAUS6", "file": "20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_onsets.csv"},
    {"date": "20260126_USP28", "position": "05", "target": "siLUC1siHAUS6", "file": "20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_onsets.csv"},
    {"date": "20260126_USP28", "position": "06", "target": "siLUC1siHAUS6", "file": "20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_onsets.csv"},
    {"date": "20260126_USP28", "position": "07", "target": "siLUC1siHAUS6", "file": "20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_onsets.csv"},
    {"date": "20260126_USP28", "position": "08", "target": "siUSP28siHAUS6", "file": "20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_onsets.csv"},
    {"date": "20260126_USP28", "position": "09", "target": "siUSP28siHAUS6", "file": "20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_onsets.csv"},
    {"date": "20260126_USP28", "position": "10", "target": "siUSP28siHAUS6", "file": "20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_onsets.csv"},
    {"date": "20260126_USP28", "position": "11", "target": "siUSP28siHAUS6", "file": "20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_onsets.csv"},
    {"date": "20260126_USP28", "position": "12", "target": "siUSP28siHAUS6", "file": "20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_onsets.csv"},
    {"date": "20260126_USP28", "position": "13", "target": "siUSP28siHAUS6", "file": "20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_onsets.csv"},
    {"date": "20260126_USP28", "position": "14", "target": "siUSP28siHAUS6", "file": "20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_onsets.csv"},

    
    {"date": "20260205_USP28", "position": "01", "target": "siLUC", "file": "20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_onsets.csv"},
    {"date": "20260205_USP28", "position": "02", "target": "siLUC", "file": "20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_onsets.csv"},
    {"date": "20260205_USP28", "position": "03", "target": "siLUCsiHAUS6", "file": "20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_onsets.csv"},
    {"date": "20260205_USP28", "position": "04", "target": "siLUCsiHAUS6", "file": "20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_onsets.csv"},
    {"date": "20260205_USP28", "position": "05", "target": "siLUCsiHAUS6", "file": "20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_onsets.csv"},
    {"date": "20260205_USP28", "position": "06", "target": "siLUCsiHAUS6", "file": "20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_onsets.csv"},
    {"date": "20260205_USP28", "position": "07", "target": "siLUCsiHAUS6", "file": "20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_onsets.csv"},
    {"date": "20260205_USP28", "position": "08", "target": "siUSP28HAUS6", "file": "20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_onsets.csv"},
    {"date": "20260205_USP28", "position": "09", "target": "siUSP28HAUS6", "file": "20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_onsets.csv"},
    {"date": "20260205_USP28", "position": "10", "target": "siUSP28HAUS6", "file": "20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_onsets.csv"},
    {"date": "20260205_USP28", "position": "11", "target": "siUSP28HAUS6", "file": "20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_onsets.csv"},
    {"date": "20260205_USP28", "position": "12", "target": "siUSP28HAUS6", "file": "20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_onsets.csv"},
    {"date": "20260205_USP28", "position": "13", "target": "siUSP28HAUS6", "file": "20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_onsets.csv"},
    {"date": "20260205_USP28", "position": "14", "target": "siUSP28HAUS6", "file": "20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_onsets.csv"},

    {"date": "20260306_USP28", "position": "01", "target": "siLUC1", "file": "20260306_IM_H2B-GFP_MitoStop_RNAiUSP28_onsets.csv"},
    {"date": "20260306_USP28", "position": "02", "target": "siLUC1", "file": "20260306_IM_H2B-GFP_MitoStop_RNAiUSP28_onsets.csv"},
    {"date": "20260306_USP28", "position": "03", "target": "siLUC1", "file": "20260306_IM_H2B-GFP_MitoStop_RNAiUSP28_onsets.csv"},
    {"date": "20260306_USP28", "position": "04", "target": "siLUC1", "file": "20260306_IM_H2B-GFP_MitoStop_RNAiUSP28_onsets.csv"},
    {"date": "20260306_USP28", "position": "05", "target": "siLUC1siHAUS6", "file": "20260306_IM_H2B-GFP_MitoStop_RNAiUSP28_onsets.csv"},
    {"date": "20260306_USP28", "position": "06", "target": "siLUC1siHAUS6", "file": "20260306_IM_H2B-GFP_MitoStop_RNAiUSP28_onsets.csv"},
    {"date": "20260306_USP28", "position": "07", "target": "siLUC1siHAUS6", "file": "20260306_IM_H2B-GFP_MitoStop_RNAiUSP28_onsets.csv"},
    {"date": "20260306_USP28", "position": "08", "target": "siUSP28siHAUS6", "file": "20260306_IM_H2B-GFP_MitoStop_RNAiUSP28_onsets.csv"},
    {"date": "20260306_USP28", "position": "09", "target": "siUSP28siHAUS6", "file": "20260306_IM_H2B-GFP_MitoStop_RNAiUSP28_onsets.csv"},
    {"date": "20260306_USP28", "position": "10", "target": "siUSP28siHAUS6", "file": "20260306_IM_H2B-GFP_MitoStop_RNAiUSP28_onsets.csv"},
    {"date": "20260306_USP28", "position": "11", "target": "siUSP28siHAUS6", "file": "20260306_IM_H2B-GFP_MitoStop_RNAiUSP28_onsets.csv"},
    {"date": "20260306_USP28", "position": "12", "target": "siUSP28siHAUS6", "file": "20260306_IM_H2B-GFP_MitoStop_RNAiUSP28_onsets.csv"},

]

# Generate and combine dataframes
df_onsets_list = []

for info in onsets_info:
    folder = f"{root}/{info['date']}/Position_{info['position']}_si{info['target']}/Concatenated"
    filepath = os.path.join(folder, info["file"])
    df_single = getonsets(filepath, info["position"], info["date"])
    df_onsets_list.append(df_single)

for info in onsets_info2:
    base_folder = f"{root}/{info['date']}/Position_{info['position']}_{info['target']}"

    folder = None
    for f in os.listdir(base_folder):
        if f.lower() == "max":
            folder = os.path.join(base_folder, f)
            break

    filepath = os.path.join(folder, info["file"])
    df_single = getonsets(filepath, info["position"], info["date"])
    
    df_onsets_list.append(df_single)

df_onsets = pd.concat(df_onsets_list, ignore_index = True)

df_onsets["Position"] = df_onsets.Position.astype(int)
df_onsets["Dataset"] = df_onsets.Dataset.str[:8]
df_onsets["Dataset"] = df_onsets.Dataset.astype(int)
df_onsets["Unique_Source_ID"] = df_onsets.Dataset.astype(str) + df_onsets.Position.astype(str) + df_onsets.Source_ID.astype(str)

In [940]:
# Find duplicate rows
unique_source_ids = df_onsets[["Unique_Source_ID"]]
duplicates = unique_source_ids[unique_source_ids.duplicated()]
print(duplicates)

Empty DataFrame
Columns: [Unique_Source_ID]
Index: []


In [941]:
df_onsets.to_csv(outputFolder + "DeBug_Onsets.csv")
df_onsets.columns

Index([' ', 'Label', 'X', 'Y', 'Frame_onset', 'Source_ID', 'Position',
       'Dataset', 'Unique_Source_ID'],
      dtype='object')

In [942]:
# Find duplicate rows in main dataframe
unique_source_ids = df[["Unique_Source_ID"]]
duplicates = unique_source_ids[unique_source_ids.duplicated()]
print(duplicates)
duplicates.to_csv(outputFolder + "DeBug_Duplicates.csv")

     Unique_Source_ID
1290    2025120463037
1291    2025120462755
1292    2025120462708
1293    2025120462038
1294    2025120462177
...               ...
1347    2025120462089
1348    2025120462673
1349    2025120463093
1350    2025120462641
1351    2025120462194

[62 rows x 1 columns]


In [943]:
df.shape

(4478, 37)

In [944]:
df = df.drop(columns = ['Unnamed: 0', 'Seen_sister', 'Seen_granny', 'EndPoint_ID'])


In [945]:
# check if any columns contain rows that contain lists

problem_cols = df.columns[df.applymap(type).eq(list).any()]
print(problem_cols)

Index([], dtype='object')


C:\Users\tkletter\AppData\Local\Temp\ipykernel_22000\1317963114.py:3: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  problem_cols = df.columns[df.applymap(type).eq(list).any()]


In [946]:
df = df.drop_duplicates()
df.shape

(4416, 33)

In [947]:
df["Frame"] = df.Frame.astype(int) + 1 #otherwise the trackmate and mitotic timing frames are not aligned

df_onsets["Dataset"] = df_onsets["Dataset"].astype(str)

df = df.merge(df_onsets, how = "inner") 

In [948]:
df.columns

Index(['Track_ID', 'Source_ID', 'Splitting_event', 'Lineage',
       'Track_Coordinate_X', 'Track_Coordinate_Y', 'Frame', 'Position',
       'Dataset', 'Condition', 'Generation', 'Mother_ID', 'Grandmother_ID',
       'Sister_ID', 'Daughters_ID', 'Number_of_daughters_in_mitosis',
       'Mother_Number_of_daughters_in_mitosis', 'Cell_Cycle_mins',
       'Cell_Cycle_hours', 'N_dead_offspring', 'N_leaving_offspring',
       'Laggings', 'Massive Missegregation', 'DNA bridge', 'Misaligned', 'MN',
       'Multipolar Division', 'Cytokinesis Failure', 'Slippage',
       'Mitotic Death', 'Comments', 'any_true', 'Unique_Source_ID', ' ',
       'Label', 'X', 'Y', 'Frame_onset'],
      dtype='object')

In [949]:
#df.to_csv(outputFolder + "DeBug_MitoticStopwatch_Augmin.csv")

In [950]:
def getInterphaseDuration(x, dataframe = df):
    # should be the same as cell_cycle - mitotic duration
    daughter_time = x.Frame_onset
    if x.Generation < 2:
        return None
    else:
        mother_row = dataframe.loc[
            (dataframe.Dataset == x.Dataset) &
            (dataframe.Position == x.Position) &
            (dataframe.Source_ID == x.Mother_ID) # ADD TRACK ID????
        ]
       # print(mother_row[["Dataset", "Position", "Source_ID"]])
        if mother_row["Frame"].shape[0] == 1:   
            mother_time = mother_row["Frame"].item()
            interphase_duration = (int(daughter_time) - int(mother_time)) * 7
            return interphase_duration
        else:
            return None


def getMitoticDurationMother(x, dataframe = df):
    if x.Generation < 2:
        return None
    else:
        mother_row = dataframe.loc[
            (dataframe.Dataset == x.Dataset) &
            (dataframe.Position == x.Position) &
            (dataframe.Source_ID == x.Mother_ID)
        ]
        
        if len(mother_row) == 1:
            return mother_row.iloc[0].Mitotic_duration_mins
        else:
            return None

def getMetaphaseArrest(x):
    if x < 30:
        return "000-030 min"
    elif (x < 60) & (x > 30):
        return "030-060"
    elif (x < 90) & (x > 60):
        return "060-090"
    elif (x < 120) & (x > 90):
        return "090-120"
    else:
        return "> 120 min"

def getMetaphaseArrest_mother(x):
    if x > 0:
        if x < 30:
            return "000-030 min"
        elif (x < 60) & (x > 30):
            return "030-060"
        elif (x < 90) & (x > 60):
            return "060-090"
        elif (x < 120) & (x > 90):
            return "090-120"
        else:
            return "> 120 min"
    else:
        pass

df["Frames_in_mitosis"] = (df.Frame - df.Frame_onset.astype(int))
df["Mitotic_duration_mins"] = df.Frames_in_mitosis * 7

df["Interphase_duration_mins"] = df.apply(getInterphaseDuration, axis = 1) 
df["Interphase_duration_hrs"] = df.Interphase_duration_mins / 60
df["Mitotic_duration_hrs"] = df.Mitotic_duration_mins / 60

df["Mother_Mitotic_duration_mins"] = df.apply(getMitoticDurationMother, axis = 1)
df["Mother_arrested"] = df.Mother_Mitotic_duration_mins.apply(getMetaphaseArrest_mother)
df["Metaphase_arrested"] = df.Mitotic_duration_mins.apply(getMetaphaseArrest)
df["Frame_onset_hrs"] = df.Frame_onset * 7 / 60 # Augmin depletion time at NEB
df["Frame_hrs"] = df.Frame * 7 / 60 # Augmin depletion time at anaphase

# Create Bins (one bin for every 12 h of depletion, binning cells with the same onset of mitotis)
interval_range_depletion = pd.interval_range(start = 12, freq = 24, end = 84) 
df['Augmin_depletion_bin'] = pd.cut(df['Frame_onset_hrs'], bins = interval_range_depletion).astype(str)

# Create Bins (one bin for every 30 min of mitotic duration) # kind of the same as "Metaphase_arrested"
interval_range_mitosis = pd.interval_range(start = 0, freq = 30, end = 900) 
df['Mitotic_duration_bin'] = pd.cut(df['Mitotic_duration_mins'], bins = interval_range_mitosis).astype(str)

df["Unique_Track_ID"] = df.Dataset.astype(str) + df.Position.astype(str) + df.Track_ID.astype(str)
df["Unique_Source_ID"] = df.Dataset.astype(str) + df.Position.astype(str) + df.Source_ID.astype(str)

# Drop annotations with mistakes
df = df.drop(df[df.Cell_Cycle_hours < 1].index)
df = df.drop(df[df.Mitotic_duration_mins < 0].index)
df = df.drop(df[df.Mother_Mitotic_duration_mins < 0].index)

df.head()

,Track_ID,Source_ID,Splitting_event,Lineage,Track_Coordinate_X,Track_Coordinate_Y,Frame,Position,Dataset,Condition,...,Interphase_duration_hrs,Mitotic_duration_hrs,Mother_Mitotic_duration_mins,Mother_arrested,Metaphase_arrested,Frame_onset_hrs,Frame_hrs,Augmin_depletion_bin,Mitotic_duration_bin,Unique_Track_ID
0,3,2153,True,2153,874.632911,867.175122,4,2,20250724,siLUC1,...,NaN,0.466667,NaN,None,000-030 min,0.000000,0.466667,nan,"(0, 30]",2025072423
1,14,2989,True,2989,1014.949828,559.472276,105,2,20250724,siLUC1,...,NaN,0.933333,NaN,None,030-060,11.316667,12.250000,nan,"(30, 60]",20250724214
2,15,3052,True,3052,1051.410130,420.260216,111,2,20250724,siLUC1,...,NaN,0.816667,NaN,None,030-060,12.133333,12.950000,"(12, 36]","(30, 60]",20250724215
3,0,2005,True,2005,666.091036,713.323699,118,2,20250724,siLUC1,...,NaN,1.166667,NaN,None,060-090,12.600000,13.766667,"(12, 36]","(60, 90]",2025072420
4,2,2121,True,2121,499.809964,637.640952,119,2,20250724,siLUC1,...,NaN,1.400000,NaN,None,060-090,12.483333,13.883333,"(12, 36]","(60, 90]",2025072422


In [951]:
df.Dataset.unique()

array(['20250724', '20250807', '20250814', '20251204', '20251215',
       '20260126', '20260205', '20260306'], dtype=object)

In [952]:
df.groupby("Dataset").Position.unique()

Dataset
20250724                           [2, 3, 4, 5, 6, 7, 8, 9]
20250807                           [2, 3, 4, 5, 6, 7, 8, 9]
20250814                              [2, 3, 4, 5, 6, 7, 8]
20251204                [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
20251215    [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
20260126    [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
20260205    [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
20260306            [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
Name: Position, dtype: object

In [953]:
df.groupby(["Dataset", 'Condition']).Splitting_event.value_counts()

Dataset   Condition       Splitting_event
20250724  siHAUS6         True               185
          siLUC1          True               200
20250807  siHAUS6         True               138
          siLUC1          True               142
20250814  siHAUS6         True               139
          siLUC1          True               103
20251204  siHAUS6         True               457
          siLUC1          True               207
20251215  siHAUS6         True               215
          siLUC1          True               171
          siMock          True                85
20260126  siLUC1          True               181
          siLUC1siHAUS6   True               334
          siUSP28siHAUS6  True               445
20260205  siLUC1          True               107
          siLUC1siHAUS6   True               280
          siUSP28siHAUS6  True               392
20260306  siLUC1          True               227
          siLUC1siHAUS6   True               169
          siUSP28siHAUS6  T

In [954]:
# Correct daughter deaths with mitotic deaths

def correct_death(x):
    if x["Mitotic Death"] == True:
        return x.N_dead_offspring - 2
    else:
        return x.N_dead_offspring

def death_bool(x):
    if x > 0:
        return True
    else:
        return False

df["N_dead_offspring"] = df.apply(correct_death, axis = 1) 
df["N_dead_offspring_bool"] = df.N_dead_offspring.apply(death_bool)

In [955]:
df.to_csv(outputFolder + "MainDataFrame_MitoticStopwatch_Augmin.csv")

print("Finished.")

Finished.
